# 🧹 Data Preprocessing with Scikit-Learn

A practical Machine Learning preprocessing workflow:

**Import Data → Inspect → Handle Missing Values → Split Data → Encode → Transform → Train Model → Evaluate**

### 🎯 Topics Covered
- Loading tabular data
- Exploring missing values
- Handling missing values with Pandas
- Separating features and target
- Train/test split
- `SimpleImputer`
- `OneHotEncoder`
- `ColumnTransformer`
- Random Forest Regression

### 🛠️ Tools
Python • NumPy • Pandas • Scikit-Learn

In [ ]:
import numpy as np
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!\n

## 1. Import Data

We use a small car-sales dataset containing numerical and categorical
features, with some missing values intentionally included.

In [ ]:
car_sales_missing = pd.DataFrame({
    "Make": ["Toyota", "Honda", "BMW", "Audi", np.nan, "Toyota", "Honda", "BMW", "Audi", "Toyota"],
    "Colour": ["Blue", "Red", "Black", np.nan, "White", "Green", "Blue", "Black", "Red", np.nan],
    "Odometer (KM)": [15000, 22000, np.nan, 18000, 25000, np.nan, 32000, 27000, 19000, 21000],
    "Doors": [4, 4, 2, np.nan, 4, 4, np.nan, 2, 4, 4],
    "Price": [18000, 21000, 35000, 30000, 19500, 17000, 22000, 34000, 29000, 20000]
})

car_sales_missing.head()

     Make Colour  Odometer (KM)  Doors  Price
0  Toyota   Blue        15000.0    4.0  18000
1   Honda    Red        22000.0    4.0  21000
2     BMW  Black            NaN    2.0  35000
3    Audi    NaN        18000.0    NaN  30000
4     NaN  White        25000.0    4.0  19500

In [ ]:
print("Dataset shape:", car_sales_missing.shape)

print("\nMissing values:")
print(car_sales_missing.isna().sum())

Dataset shape: (10, 5)

Missing values:
Make             1
Colour           2
Odometer (KM)    2
Doors            2
Price            0
dtype: int64


## 2. Handle Missing Values with Pandas

For this demonstration:

- `Make` → `"missing"`
- `Colour` → `"missing"`
- `Odometer (KM)` → mean
- `Doors` → 4

The target (`Price`) has no missing values.

In [ ]:
car_sales_clean = car_sales_missing.copy()

car_sales_clean["Make"] = car_sales_clean["Make"].fillna("missing")
car_sales_clean["Colour"] = car_sales_clean["Colour"].fillna("missing")
car_sales_clean["Odometer (KM)"] = car_sales_clean["Odometer (KM)"].fillna(
    car_sales_clean["Odometer (KM)"].mean()
)
car_sales_clean["Doors"] = car_sales_clean["Doors"].fillna(4)

print("Missing values after Pandas imputation:")
print(car_sales_clean.isna().sum())

car_sales_clean.head()

Missing values after Pandas imputation:
Make             0
Colour           0
Odometer (KM)    0
Doors            0
Price            0
dtype: int64


      Make   Colour  Odometer (KM)  Doors  Price
0   Toyota     Blue        15000.0    4.0  18000
1    Honda      Red        22000.0    4.0  21000
2      BMW    Black        22375.0    2.0  35000
3     Audi  missing        18000.0    4.0  30000
4  missing    White        25000.0    4.0  19500

## 3. Separate Features and Target

`X` contains the input features.

`y` contains the value we want to predict.

In [ ]:
X = car_sales_missing.drop("Price", axis=1)
y = car_sales_missing["Price"]

print("Features:", list(X.columns))
print("Target: Price")

Features: ['Make', 'Colour', 'Odometer (KM)', 'Doors']
Target: Price


## 4. Train/Test Split

We split the data before fitting preprocessing steps so that the test set
does not influence the learned preprocessing parameters.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8
Testing samples: 2


## 5. Impute Missing Values with `SimpleImputer`

Different columns use different strategies:

- Categorical columns → most frequent value
- Numerical column → mean
- Doors → most frequent value

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour"]
door_feature = ["Doors"]
numeric_features = ["Odometer (KM)"]

cat_imputer = SimpleImputer(strategy="most_frequent")
door_imputer = SimpleImputer(strategy="most_frequent")
num_imputer = SimpleImputer(strategy="mean")

imputer = ColumnTransformer([
    ("cat", cat_imputer, categorical_features),
    ("doors", door_imputer, door_feature),
    ("numeric", num_imputer, numeric_features)
])

filled_X_train = imputer.fit_transform(X_train)
filled_X_test = imputer.transform(X_test)

print("Training data transformed successfully!")
print("Training shape:", filled_X_train.shape)
print("Testing shape:", filled_X_test.shape)

Training data transformed successfully!
Training shape: (8, 4)
Testing shape: (2, 4)


## 6. One-Hot Encode Categorical Features

Machine Learning models need numerical input. `OneHotEncoder` converts
categorical columns into numerical indicator columns.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot = OneHotEncoder(handle_unknown="ignore")

transformer = ColumnTransformer([
    (
        "one_hot",
        one_hot,
        categorical_features + door_feature
    )
], remainder="passthrough")

# Convert the imputed arrays back to DataFrames so the column names
# remain explicit before encoding.
train_filled_df = pd.DataFrame(
    filled_X_train,
    columns=["Make", "Colour", "Doors", "Odometer (KM)"]
)

test_filled_df = pd.DataFrame(
    filled_X_test,
    columns=["Make", "Colour", "Doors", "Odometer (KM)"]
)

transformed_X_train = transformer.fit_transform(train_filled_df)
transformed_X_test = transformer.transform(test_filled_df)

print("Encoded training shape:", transformed_X_train.shape)
print("Encoded testing shape:", transformed_X_test.shape)

Encoded training shape: (8, 11)
Encoded testing shape: (2, 11)


## 7. Train a Machine Learning Model

After preprocessing, the data is ready for a regression model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(transformed_X_train, y_train)

predictions = model.predict(transformed_X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print("Model trained successfully!")
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.3f}")

Model trained successfully!
MAE : 775.00
RMSE: 790.57
R²  : 0.986


## 8. Actual vs Predicted Prices

In [ ]:
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": np.round(predictions, 2)
})

results

 Actual Price  Predicted Price
        20000          20775.0
        21000          19450.0

## 🎯 Key Takeaways

- Missing values must be handled before model training.
- Features and target should be separated.
- Train/test splitting should happen before fitting preprocessing.
- `SimpleImputer` handles missing values.
- `OneHotEncoder` converts categorical data into numerical features.
- `ColumnTransformer` applies different transformations to different columns.
- The same fitted preprocessing steps are applied to the test set.
- After preprocessing, the transformed data can be passed to an ML model.

### 🔄 Workflow

**Raw Data → Inspect → Impute → Split → Encode → Transform → Train → Evaluate**

## 🚀 Next Step

**Regression:** Linear Regression, Random Forest Regression, MAE, MSE, RMSE and R².